# Function 5


In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+0

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5], [0.210195, 0.849014, 0.874271, 0.875537], [0.241981, 0.847887, 0.878175, 0.876911], [0.230056, 0.841143, 0.873217, 0.887638], [0.241981, 0.847887, 0.878175, 0.876911]]
additionalOutputs = [np.float64(32.0025), np.float64(1060.3898045443261), np.float64(1083.0108574790386), np.float64(1081.4793716878269), np.float64(1083.0108574790386)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.5        0.5        0.5        0.5       ]
 [0.210195   

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# --- Latin Hypercube Sampling (LHS) ---
n_samples = 12500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(12500000, 4)


In [5]:
print(x_grid[:10])

[[0.07564682 0.19730556 0.61438713 0.9059265 ]
 [0.29252671 0.03016936 0.6251265  0.07986546]
 [0.79961855 0.61025564 0.27169621 0.86352841]
 [0.94375003 0.14954025 0.35439468 0.42597542]
 [0.00237004 0.12494751 0.09195369 0.27859139]
 [0.93379658 0.02449221 0.97306184 0.30644521]
 [0.05330282 0.53045086 0.98159828 0.84677248]
 [0.32496575 0.80649195 0.67306674 0.55027   ]
 [0.18394533 0.21741149 0.4257746  0.58111678]
 [0.76197927 0.99845388 0.94852926 0.00507875]]


# Bayesian Optimisation with UCB applied

In [7]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
#beta = 1.96
#beta = 0.5
beta = 0

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.24198198 0.84788742 0.87817589 0.87691181]
